#### General guidance

This serves as a template which will guide you through the implementation of this task. It is advised
to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [1]:
import numpy as np
import pandas as pd

# Add any additional imports here (however, the task is solvable without using 
# any additional imports)
# import ...
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV

 #### Loading data

In [2]:
data = pd.read_csv("train.csv")
y = data["y"].to_numpy()
data = data.drop(columns=["Id", "y"])
# print a few data samples
print(data.head())
X = data.to_numpy()

     x1    x2    x3    x4    x5
0  0.02  0.05 -0.09 -0.43 -0.08
1 -0.13  0.11 -0.08 -0.29 -0.03
2  0.08  0.06 -0.07 -0.41 -0.03
3  0.02 -0.12  0.01 -0.43 -0.02
4 -0.14 -0.12 -0.08 -0.02 -0.08


#### Transform data

In [4]:
"""
Transform the 5 input features of matrix X (x_i denoting the i-th component of X) 
into 21 new features phi(X) in the following manner:
5 linear features: phi_1(X) = x_1, phi_2(X) = x_2, phi_3(X) = x_3, phi_4(X) = x_4, phi_5(X) = x_5
5 quadratic features: phi_6(X) = x_1^2, phi_7(X) = x_2^2, phi_8(X) = x_3^2, phi_9(X) = x_4^2, phi_10(X) = x_5^2
5 exponential features: phi_11(X) = exp(x_1), phi_12(X) = exp(x_2), phi_13(X) = exp(x_3), phi_14(X) = exp(x_4), phi_15(X) = exp(x_5)
5 cosine features: phi_16(X) = cos(x_1), phi_17(X) = cos(x_2), phi_18(X) = cos(x_3), phi_19(X) = cos(x_4), phi_20(X) = cos(x_5)
1 constant feature: phi_21(X)=1

Parameters
----------
X: matrix of floats, dim = (700,5), inputs with 5 features

Compute
----------
X_transformed: array of floats: dim = (700,21), transformed input with 21 features
"""
X_transformed = np.zeros((700, 21))
# TODO: Enter your code here

X_transformed[:,0:5] = X[:,0:5]
X_transformed[:,5:10] = np.power(X[:,0:5],2)
X_transformed[:,10:15] = np.exp(X[:,0:5])
X_transformed[:,15:20] = np.cos(X[:,0:5])
X_transformed[:,20] = np.ones((700,))

"""tmp = pd.DataFrame(np.column_stack((X_transformed, y)), columns=["x1","x2","x3","x4","x5","x6","x7","x8","x9","x10","x11","x12","x13","x14","x15","x16","x17","x18","x19","x20","x21","y"])
tmp1 = tmp[["x1","y"]]
sns.relplot(data=tmp1,x="x1",y="y")
tmp2 = tmp[["x2","y"]]
sns.relplot(data=tmp2,x="x2",y="y")
tmp3 = tmp[["x3","y"]]
sns.relplot(data=tmp3,x="x3",y="y")
tmp4 = tmp[["x4","y"]]
sns.relplot(data=tmp4,x="x4",y="y")
tmp5 = tmp[["x5","y"]]
sns.relplot(data=tmp5,x="x5",y="y")
tmp6 = tmp[["x6","y"]]
sns.relplot(data=tmp6,x="x6",y="y") 
tmp11 = tmp[["x11","y"]]
sns.relplot(data=tmp11,x="x11",y="y") 
tmp16 = tmp[["x16","y"]]
sns.relplot(data=tmp16,x="x16",y="y") 
tmp21 = tmp[["x21","y"]]
sns.relplot(data=tmp21,x="x21",y="y")"""

assert X_transformed.shape == (700, 21)

#### Fit data

In [45]:
""" Use the transformed data points X_transformed and fit the linear regression on this 
transformed data. Finally, compute the weights of the fitted linear regression. 

Parameters
----------
X_transformed: array of floats: dim = (700,21), transformed input with 21 features
y: array of floats, dim = (700,), input labels)

Compute
----------
w: array of floats: dim = (21,), optimal parameters of linear regression """
w = np.zeros((21,))
# TODO: Enter your code here


#Manual finding of optimal lambda, use Ridge Regression as 
"""
lambdas = np.linspace(0,300,3000) 
#we have such little data that I think a smaller test size is better
#the downside being that the error estimate is worse, thus there's no point in trying too many lambdas
opt_lam = -1.0
min_RMSE = 100.0
random_states = [42,2,10,4,88,33,52,11,100,5]
done = False #manual condition to avoid running the somewhat long for-loops every time
if not done: 
    for lam in lambdas: 
        RMSE_mat = [0,0,0,0,0,0,0,0,0,0]
        for it in range(10): #get a median over 10 different training data splits
            X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.10, random_state=random_states[it]) 
            reg = Ridge(alpha = lam, fit_intercept = False).fit(X_train, y_train)
            y_pred = reg.predict(X_test)
            RMSE_mat[it] = mean_squared_error(y_test,y_pred, squared=False) 
        RMSE = np.median(RMSE_mat)
        if(RMSE < min_RMSE):
            min_RMSE = RMSE
            opt_lam = lam
    print("Best lambda: ",opt_lam," with RMSE ",min_RMSE)
else:
    opt_lam = 203.06768922974325

#train on entire data set (because its so small, that is quite important)
w = Ridge(alpha = opt_lam, fit_intercept = False).fit(X_transformed,y).coef_
print("weights: ",w)"""

#2nd attempt finding optimal lambda
done = False
opt_lam = -1
if not done:
    min_val = 1000
    kf = KFold(n_splits=10,shuffle=False)
    #we have previously determined that lambdas < 1 and > 300 are worse
    for lam in np.linspace(0,300,3000):
        ridge_reg = Ridge(alpha=lam,fit_intercept = False)
        scores = cross_val_score(ridge_reg,X_transformed,y,scoring="neg_root_mean_squared_error",cv=kf)
        mean = (-1)*np.mean(scores)
        if mean < min_val:
            min_val = mean
            opt_lam = lam
        #print(scores,mean)
    print(opt_lam,min_val)
    #plt.plot(means)
    #plt.show()
else :
    opt_lam = 42.8
#train on entire data set
w = Ridge(alpha=opt_lam, fit_intercept = False).fit(X_transformed,y).coef_
print(w)

#With training, test data split
"""
lambdas = np.linspace(40,50,100) 
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.10, random_state=42) 
reg = RidgeCV(alphas = lambdas, fit_intercept = False).fit(X_train, y_train)
print("Alpha: ",reg.alpha_)
print("Weights: ",reg.coef_)
y_pred = reg.predict(X_test)
RMSE = mean_squared_error(y_test,y_pred, squared=False) 
print("RMSE: ",RMSE)
#still train on entire data set, but use best alpha/lambda
w = Ridge(alpha = reg.alpha_, fit_intercept = False).fit(X_transformed,y).coef_
"""

#Manual cross validation
"""
lambdas = [45,46,47,48,49,50]
n_folds = 10
RMSE_mat = np.zeros((n_folds, len(lambdas)))

kf  = KFold(n_splits = n_folds,random_state = None)
for i,lam in enumerate(lambdas): 
    for j, (train_index, test_index) in enumerate (kf.split(X_transformed)): 
        X_train, X_test = X_transformed[train_index], X_transformed[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        ridge_reg = Ridge(alpha=lam,fit_intercept = False) #as we only pass the coeffs/weights, we need the bias b to be 0 
        w = ridge_reg.fit(X_train,y_train).coef_

        y_pred = np.dot(X_test,w)
        RMSE_mat[j][i] = mean_squared_error(y_test,y_pred, squared=False)

avg_RMSE = np.mean(RMSE_mat, axis=0)
#print(avg_RMSE)

#Use entire test set to compute weights
reg = Ridge(alpha=45,fit_intercept = False).fit(X_transformed,y)
w = reg.coef_
"""

#No training, test data split
#aka no idea how this performs (:
"""reg = RidgeCV(alphas = lambdas, fit_intercept = False).fit(X_transformed, y)
w = reg.coef_"""

#So now lets try Lasso
# without splitting training data either because eh
"""reg = LassoCV(alphas = lambdas, fit_intercept = False,max_iter=100000,n_alphas = 10000).fit(X_transformed,y) #increased max iterations until no warning
w = reg.coef_
print(w)"""

assert w.shape == (21,)

42.81427142380794 1.9517422496340893
[ 0.062554   -0.0916428  -0.13715239  0.19542675  0.05456034 -0.04989471
  0.01608556  0.02431581 -0.08050137  0.00289955 -0.5603188  -0.6818757
 -0.72377572 -0.43886104 -0.54231373 -0.5727341  -0.60551709 -0.60960759
 -0.5578901  -0.598821   -0.59752977]


In [46]:
# Save results in the required format
np.savetxt("./results.csv", w, fmt="%.12f")